In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:


data =[
    (1,"C001","Laptop","50000","2024-01-01"),
    (2,"C002","Mobile",None,"2024-01-02"),
    (3,"C003","Tablet","20000","2024-01-03"),
    (4,"C004","Laptop","55000","2024-01-04"),
    (5,"C005","Headphones",None,"2024-01-05"),
    (6,"C006","Camera","30000","2024-01-07"),
    (7,"C007","Mobile","18000","2024-01-07"),
    (8,"C008","Watch","8000","2024-01-07")

]

columns = ["order_id","customer_id","product","amount","updated_date"]

df= spark.createDataFrame(data,columns)

In [0]:
df.printSchema()

In [0]:
df.fillna({"amount":0}).display()

In [0]:
df=df.withColumn("amount",df.amount.cast("long"))
df.printSchema()

In [0]:
df=df.fillna({"amount":1000})
df.display()

In [0]:
df= df.withColumn("updated_date",df.updated_date.cast("date"))
df.printSchema()

In [0]:
df=df.withColumn("bonus",col("amount")*0.1)
df.display()

In [0]:
df.display()

In [0]:
df=df.withColumn("category",
                 when(col("amount")>20000,"High").otherwise("Low"))
df.display()

In [0]:
def amount_bucket(amount):
    if amount > 30000:
        return "High"
    elif 30000>=amount>= 10000:
        return "Medium"
    else:
        return "Low"
    

In [0]:
amount_bucket_udf = udf(amount_bucket, StringType())

In [0]:
df=df.withColumn("amount_bucket",amount_bucket_udf(df.amount))
df.display()

In [0]:
df.write.mode("overwrite").parquet("/Volumes/workspace/default/endtoendpipelinebysravansir")

In [0]:
existing_df = spark.read.parquet("/Volumes/workspace/default/endtoendpipelinebysravansir")
existing_df.display()

In [0]:
last_loaded_date = existing_df.select(
    max("updated_date")
).collect()[0][0]

print("Last loaded date:", last_loaded_date)

In [0]:
data2 = [
    (3, "C003", "Tablet", "22000", "2024-01-06")
]
df_2 = spark.createDataFrame(data2,columns)
df_2.display()

In [0]:
df_2=df_2.withColumn("amount",df_2.amount.cast("long"))\
    .withColumn("updated_date",df_2.updated_date.cast("date"))

In [0]:
df_2.display()

In [0]:
new_df = df_2
new_df.display()

In [0]:
incremental_df= new_df.filter(
    col("updated_date") > last_loaded_date
)

In [0]:
incremental_df.display()

In [0]:
filtered_existing = existing_df.join(incremental_df,
                                     on="order_id",
                                     how="left_anti")

In [0]:
filtered_existing.display()

In [0]:
# Apply same transformations to incremental_df
incremental_df = incremental_df.withColumn("bonus", col("amount") * 0.1) \
    .withColumn("category", when(col("amount") > 20000, "High").otherwise("Low")) \
    .withColumn("amount_bucket", amount_bucket_udf(col("amount")))

final_df = filtered_existing.union(incremental_df)
final_df.display()

In [0]:
incremental_df.printSchema()

In [0]:
final_df = filtered_existing.union(incremental_df)
final_df.display()

In [0]:
final_df.write.mode("overwrite").parquet("/Volumes/workspace/default/endtoendpipelinebysravansir")